# Phase 2: Data Cleaning, Preprocessing & Feature Engineering Foundation
## AI-Powered Fake Job and Recruitment Scam Detection System

**Project Context**: Final-Year B.Tech CSE (AI & ML) Academic Project  
**Objective**: Construct a modular, leak-free, reproducible preprocessing and feature-engineering pipeline that:
1. Validates and preserves raw dataset immutability.
2. Normalizes text without destroying semantic boundaries.
3. Constructs structured composite text representations with field headers.
4. Handles missing values in categorical, binary, and salary fields systematically.
5. Partitions data into a **stratified 80/20 train/test split** with zero ID overlap.
6. Fits TF-IDF (10,000 max features, n-grams 1–2) and tabular encoders **strictly only on training data**.
7. Serializes the fitted hybrid pipeline to `models/preprocessor.joblib` and exports processed matrices.

> **Academic Rule**: No machine learning model training is performed in this phase. All preprocessing artifacts are constructed for downstream baseline and advanced experiments.

In [ ]:
import sys
import json
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from scipy import sparse

from config.settings import get_config
from src.data.loader import load_raw_dataset, calculate_file_hash
from src.preprocessing.text_cleaner import clean_text_string, build_composite_text
from src.preprocessing.splitter import create_stratified_split
from src.preprocessing.pipeline import FullHybridPreprocessor
from src.preprocessing.executor import run_phase2_preprocessing

print("Preprocessing packages imported successfully.")

## 1. Verify Raw Dataset Immutability & Ingestion

In [ ]:
config = get_config()
raw_path = config.raw_dataset_path

raw_hash = calculate_file_hash(raw_path)
print(f"Raw Dataset Path : {raw_path}")
print(f"SHA-256 Hash     : {raw_hash}")

df, raw_meta = load_raw_dataset(raw_path)
print(f"Loaded shape     : {df.shape[0]:,} rows x {df.shape[1]} columns")

## 2. Text Cleaning & Structured Composite Representation Demo

In [ ]:
# Demonstrate composite text structure on sample record
sample_row = df.iloc[0]
composite_sample = build_composite_text(sample_row, text_columns=config.text_columns, lowercase=True)

print("=" * 60)
print("SAMPLE COMPOSITE TEXT REPRESENTATION (FIRST ROW)")
print("=" * 60)
print(composite_sample[:600] + "\n... [TRUNCATED] ...")
print("=" * 60)

## 3. Stratified 80/20 Train/Test Partitioning

In [ ]:
train_df, test_df, split_meta = create_stratified_split(
    df,
    target_col=config.target_column,
    id_col=config.id_column,
    test_size=0.20,
    random_state=config.random_seed,
)

print(f"Train set: {len(train_df):,} samples | Fraud rate: {split_meta['train_class_distribution']['fraud_rate_pct']:.2f}%")
print(f"Test set : {len(test_df):,} samples | Fraud rate: {split_meta['test_class_distribution']['fraud_rate_pct']:.2f}%")
print(f"Zero ID overlap between splits: {split_meta['id_overlap_count'] == 0}")

## 4. Fit Hybrid Preprocessing Pipeline Exclusively on Training Data

In [ ]:
feature_cols = [c for c in df.columns if c not in [config.target_column, config.id_column]]
X_train = train_df[feature_cols]
y_train = train_df[config.target_column].values

X_test = test_df[feature_cols]
y_test = test_df[config.target_column].values

# Initialize and fit preprocessor on training data ONLY
preprocessor = FullHybridPreprocessor(config=config)
preprocessor.fit(X_train)

# Transform both train and test sets
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print("=" * 60)
print("FEATURE MATRIX DIMENSIONALITY & SPARSITY")
print("=" * 60)
print(f"TF-IDF Vocabulary Size : {len(preprocessor.tfidf_vectorizer.vocabulary_):,} n-grams")
print(f"Total Hybrid Features   : {len(preprocessor.feature_names_):,} features")
print(f"X_train Shape           : {X_train_transformed.shape}")
print(f"X_test Shape            : {X_test_transformed.shape}")
print(f"X_train Non-Zero Count  : {X_train_transformed.nnz:,}")
print(f"X_train Sparsity        : {(1.0 - (X_train_transformed.nnz / (X_train_transformed.shape[0] * X_train_transformed.shape[1]))) * 100:.2f}%")
print("=" * 60)

## 5. Verify Processed Artifacts & Post-Execution Raw Data Hash

In [ ]:
post_hash = calculate_file_hash(raw_path)
print(f"Pre-Execution Hash  : {raw_hash}")
print(f"Post-Execution Hash : {post_hash}")
print(f"Hash Immutability Confirmed: {raw_hash == post_hash}")

print("\nPhase 2 Complete. Foundation ready for Phase 3 Model Training.")